# 01. Environment Setup and NLP Initialization
Before processing the data, we need to initialize our Natural Language Processing (NLP) toolkit. We import nltk to handle standard English stop words (words that carry no semantic meaning, like "the", "is", "at"). Furthermore, freelance marketplaces have their own specific "noise" words (e.g., "looking", "need", "hire") that appear in almost every job posting. We append this custom upwork_garbage to our stop words list to prevent our future vectorization models (like TF-IDF or Embeddings) from treating these highly frequent but useless words as important features.

In [1]:
import nltk
import json
import html
import re
from pathlib import Path
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

# Download standard English stop words
nltk.download('stopwords', quiet=True)

# Create a combined set of standard stop words and Upwork-specific noise
stop_words = set(stopwords.words('english'))
upwork_garbage = {'looking', 'need', 'expert', 'project', 'work', 'term', 'hire', 'contractor'}
stop_words = list(stop_words.union(upwork_garbage))

# 2. Project Configurations and Paths
Hardcoding paths or variables deep inside functions leads to fragile code. We define all our file paths and load external configurations (like columns to drop or skill dictionaries) at the very top. Note: Since **file** is not defined in interactive environments like Jupyter, we use *Path.cwd()* to dynamically resolve the project root based on where the notebook is launched.

In [2]:
# Resolving paths dynamically for Jupyter Notebook environment
PROJECT_ROOT = Path.cwd().parent # Adjust .parent depending on where your notebook is located
CONFIG_PATH = PROJECT_ROOT / "configs" / "upwork-jobs.json"
INPUT_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "Final_Upwork_Dataset.csv"
OUTPUT_CSV_PATH = PROJECT_ROOT / "data" / "processed"
SKILLS_PATH = PROJECT_ROOT / "data" / "raw" / "skills.json"

# Load configurations
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = json.load(f)

# Load and normalize skills dictionary
with SKILLS_PATH.open("r", encoding="utf-8") as f:
    skills = json.load(f)
    skills = list(skills)
    skills_list_lower = [skill.lower() for skill in skills]

COLUMNS_TO_DROP = config.get("COLOUMS_TO_DROP", [])
USE_STOPWORDS = config.get("USE_STOPWORDS", True)

# 3. Feature Mapping Definitions
Machine learning models require numerical inputs, but raw datasets often contain categorical or descriptive strings (e.g., "Less than 5" applicants). We define strict mapping dictionaries to convert these categorical text ranges into continuous numerical proxies. By doing this, we preserve the ordinal relationships in the data (e.g., 50+ applicants is mathematically larger than 5 to 10), which is crucial for regression or ranking algorithms.

In [3]:
# Transforming categorical applicant ranges into numerical averages
APPLICANTS_MAPPING = {
    'Less than 5': 3.0,
    '5 to 10': 7.5,
    '10 to 15': 12.5,
    '15 to 20': 17.5,
    '20 to 50': 35.0,
    '50+': 60.0
}

# Converting descriptive project durations into estimated total project hours
TIME_LIMITATION_MAPPING = {
    '3 to 6 months, Less than 30 hrs/week': 18 * 20,
    '3 to 6 months, 30+ hrs/week': 18 * 40,
    '3 to 6 months, Hours to be determined': 18 * 30,
    '3 to 6 months, not_sure': 18 * 30,

    'More than 6 months, Less than 30 hrs/week': 30 * 20,
    'More than 6 months, 30+ hrs/week': 30 * 40,
    'More than 6 months, Hours to be determined': 30 * 30,
    'More than 6 months, not_sure': 30 * 30,

    '1 to 3 months, Less than 30 hrs/week': 8 * 20,
    '1 to 3 months, 30+ hrs/week': 8 * 40,
    '1 to 3 months, Hours to be determined': 8 * 30,
    '1 to 3 months, not_sure': 8 * 30,

    'Less than 1 month, Less than 30 hrs/week': 3 * 20,
    'Less than 1 month, 30+ hrs/week': 3 * 40,
    'Less than 1 month, Hours to be determined': 3 * 30,
    'Less than 1 month, not_sure': 3 * 30
}

# Binary encoding for payment type
PAYMENT_MAPPING = {
    'Hourly': 1,
    'Fixed-price': 0,
}

# Dynamically generate category column names
CATEGORY_COLS = [f"Category_{i}" for i in range(1, 10)]

# 4. Raw Text Normalization (NLP Preprocessing)
Raw data scraped from the web is highly unstructured. It contains HTML tags, URLs, and inconsistent punctuation. If left uncleaned, semantic search models will treat <b>React</b>, React,, and react as three entirely different skills. The clean_raw_text function strips out HTML logic, removes non-alphanumeric noise, standardizes to lowercase, and optionally filters out our predefined stop words to ensure a clean text corpus for embedding generation.

In [4]:
def drop_columns(df: pd.DataFrame, columns_to_drop: list[str]) -> pd.DataFrame:
    """Safely drops columns if they exist in the dataframe."""
    existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    return df.drop(existing_columns_to_drop, axis=1)

def clean_raw_text(text: str) -> str:
    """Normalizes raw text for NLP tasks."""
    if not isinstance(text, str):
        return ""

    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)

    # Decode HTML entities and strip tags
    text = html.unescape(text)
    text = text.replace("<null>", "")
    text = re.sub(r"<[^>]+>", "", text)

    # Normalize punctuation and specific characters
    text = text.replace("/", " ").replace("-", " ")
    text = re.sub(r"[.,!?—:;\"()\[\]{}*_\r\n]", " ", text)

    # Tokenize and remove stop words
    words = text.lower().split()
    if USE_STOPWORDS:
        words = [word for word in words if word not in stop_words]

    return " ".join(words).strip()

# 5. Unifying the Target Variable (Hourly Rate Calculation)
Freelance platforms operate on a bifurcated payment model: some jobs are "Hourly" (with a low-high range) and others are "Fixed-Price" (total budget). For a recommendation engine to properly rank candidates by budget, we must normalize these disparate systems into a single metric. This function applies vectorized numpy operations (for high performance) to calculate a median hourly rate for hourly jobs, and estimates an effective hourly rate for fixed-price jobs by dividing the total budget by the estimated project hours derived from our *TIME_LIMITATION_MAPPING*.

In [5]:
def calculate_hourly_rate(df: pd.DataFrame) -> pd.Series:
    """Calculates a unified hourly rate across different payment structures."""
    # Clean string currency values to extract pure numbers
    start_clean = df["Start_rate"].astype(str).str.replace(r'[^\d.]', '', regex=True)
    end_clean = df["End_rate"].astype(str).str.replace(r'[^\d.]', '', regex=True)
    cost_clean = df["Job_Cost"].astype(str).str.replace(r'[^\d.]', '', regex=True)

    start_arr = pd.to_numeric(start_clean, errors="coerce")
    end_arr = pd.to_numeric(end_clean, errors="coerce")
    job_cost = pd.to_numeric(cost_clean, errors="coerce").fillna(0.0)

    # Map time limitations to total hours to calculate effective fixed rates
    time_limitation_clean = df["Time_Limitation"].str.strip()
    total_project_hours = time_limitation_clean.map(TIME_LIMITATION_MAPPING).fillna(120.0)

    start_is_nan = np.isnan(start_arr)
    end_is_nan = np.isnan(end_arr)

    # Logic to handle missing bounds in hourly ranges
    cond_list = [
        ~start_is_nan & ~end_is_nan,
        start_is_nan & ~end_is_nan,
        ~start_is_nan & end_is_nan
    ]
    choice_list = [
        (start_arr + end_arr) / 2, # Average of range
        end_arr / 2,               # Fallback if only max is given
        start_arr                  # Fallback if only min is given
    ]

    hourly_avg = np.select(cond_list, choice_list, default=np.nan)
    fixed_calculated_rate = job_cost / total_project_hours

    is_valid_hourly = ~np.isnan(hourly_avg) & (hourly_avg > 0)

    # Apply valid hourly rate or fallback to calculated fixed rate
    hour_rate = np.where(
        is_valid_hourly,
        hourly_avg,
        fixed_calculated_rate
    )

    return pd.Series(hour_rate, index=df.index).fillna(0.0)

# 6. The Main Preprocessing Pipeline
This function orchestrates the entire ETL (Extract, Transform, Load) workflow. It sequentially filters out rows with critical missing data (like empty descriptions), applies our NLP text cleaning, extracts skills into a standardized list, maps categorical features, and calculates our unified target variable. Wrapping this in a single function ensures our data processing is reproducible and can be seamlessly moved back into a production script.

In [6]:
def process_upwork_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Executes the full preprocessing pipeline on the raw Upwork dataset."""

    # 1. Row Filtering (Drop critical nulls)
    df = df.dropna(subset=["Description", "Job Title"])
    df = df.dropna(subset=CATEGORY_COLS, how="all")
    df = df[df["Description"].str.strip() != ""]
    df = df[df["Job Title"].str.strip() != ""]
    df = df.reset_index(drop=True)

    # 2. Text Cleaning (NLP)
    df["Job Title"] = df["Job Title"].apply(clean_raw_text)
    df["Description"] = df["Description"].apply(clean_raw_text)

    # 3. Skill Extraction (Flatten category columns into a list)
    df["skills"] = df[CATEGORY_COLS].apply(
        lambda row: [str(item).strip().lower() for item in row if pd.notna(item) and str(item).strip() != ""],
        axis=1
    )

    # 4. Feature Mapping
    df["Applicants_Num"] = df["Applicants_Num"].map(APPLICANTS_MAPPING).fillna(-1)
    df["is_hourly"] = df["Payment_type"].map(PAYMENT_MAPPING).fillna(-1)

    # Group rare countries into 'other' to reduce dimensionality
    top_countries = df["Client_Country"].value_counts().nlargest(10).index
    df["Client_Country"] = df["Client_Country"].apply(lambda x: x if x in top_countries else "other")

    # 5. Target Variable Calculation
    df["hour_rate"] = calculate_hourly_rate(df)

    # 6. Final Cleanup (Drop redundant or intermediary columns)
    columns_to_drop = CATEGORY_COLS + ["Payment_type", "Time_Limitation", "Job_Cost"]
    df = df[df["hour_rate"] > 0].reset_index(drop=True)
    df = drop_columns(df, columns_to_drop)

    return df

 # 7. Strategic Dataset Splitting for Cross-Validation and Production
To ensure rigorous machine learning evaluation and prevent Data Leakage, we split our cleaned dataset into four entirely isolated segments:
1. **CV Train Pool (70%):** The primary training dataset. Notebook `02` will use it to build the training FAISS index, and Notebook `03` will use its category statistics to generate synthetic users. XGBoost will run Cross-Validation (CV) strictly on this block.
2. **Holdout Validation Set (10%):** An untouched set used for final hyperparameter tuning and model evaluation before deployment.
3. **Holdout Test Set (10%):** A completely blind test set to simulate model behavior on unseen market data inside our ML pipeline.
4. **Website Production Database (10%):** This subset is physically exported to populate our FastAPI backend and web production FAISS index. It contains no artificial targets, simulating real-world production data.


In [7]:
print("Starting preprocessing pipeline...")

# Load Data
df_raw = pd.read_csv(INPUT_CSV_PATH)
print(f"Raw dataset loaded. Shape: {df_raw.shape}")

# Execute Pipeline
df_processed = process_upwork_dataset(df_raw)

# Global Configurations and Quality Threshold Filter
if COLUMNS_TO_DROP:
    df_processed = drop_columns(df_processed, COLUMNS_TO_DROP)

# Explicitly filter out extreme low-ball or noise hourly rates (< $4.0)
df_processed = df_processed[df_processed["hour_rate"] >= 4.0].reset_index(drop=True)
print(f"Dataset cleaned and filtered. Quality shape: {df_processed.shape}")

print("\nExecuting multi-stage dataset split...")

# Isolate Website Production Database (10%)
df_remaining, df_site = train_test_split(df_processed, test_size=0.10, random_state=42)

# Isolate Holdout Test Set (10% of total -> ~11.11% of remaining)
df_remaining, df_test = train_test_split(df_remaining, test_size=0.1111, random_state=42)

# Split the remaining 80% into CV Train Pool (70% total) and Holdout Val (10% total)
# (12.5% of remaining gives exactly 10% of total to Validation)
df_cv_train, df_val = train_test_split(df_remaining, test_size=0.125, random_state=42)

# Exporting Split Artifacts
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# Define exact output paths for our 4-way split architecture
splits = {
    "upwork_cv_train.csv": df_cv_train,
    "upwork_holdout_val.csv": df_val,
    "upwork_holdout_test.csv": df_test,
    "upwork_site_production.csv": df_site
}

print("\nSaving data split artifacts:")
for filename, split_df in splits.items():
    save_path = OUTPUT_CSV_PATH / filename
    split_df.to_csv(save_path, index=False, encoding="utf-8")
    print(f"Saved {filename:<25} | Shape: {split_df.shape}")

print("\nPreprocessing and strategic splitting completed successfully!")

# Inspect the primary training pool schema
df_cv_train.head()

Starting preprocessing pipeline...
Raw dataset loaded. Shape: (63950, 41)
Dataset cleaned and filtered. Quality shape: (32496, 13)

Executing multi-stage dataset split...

Saving data split artifacts:
Saved upwork_cv_train.csv       | Shape: (22746, 13)
Saved upwork_holdout_val.csv    | Shape: (3250, 13)
Saved upwork_holdout_test.csv   | Shape: (3250, 13)
Saved upwork_site_production.csv | Shape: (3250, 13)

Preprocessing and strategic splitting completed successfully!


,Job Title,EX_level_demand,Description,Applicants_Num,Spent($),Client_Country,Rating,Feedback_Num,Start_rate,End_rate,skills,is_hourly,hour_rate
13156,help adding html iframe within storyblok posting,Intermediate,website built uses storyblok enable team know ...,17.5,0.0,United States,NaN,0.0,$15,$30,"[html, html5]",1,22.500000
14383,kajabi custom code,Intermediate,hey kajabi landing page almost ready go live c...,35.0,10000.0,United States,4.832985,14.0,$25,$120,"[web development, kajabi]",1,72.500000
27766,appointment setter,Expert,law firms interested international expansion m...,3.0,200.0,Netherlands,4.000000,1.0,0,NaN,"[legal, scheduling, telemarketing, cold callin...",0,8.333333
2800,interior designer architect needed create plan...,Expert,hello owner small web development company purc...,7.5,10000.0,Canada,5.000000,9.0,$20.00,$35.00,"[3d modeling, 3d rendering, interior design, a...",1,27.500000
22616,copywriter creative agency landing page,Intermediate,simple landing page certain paragraphs headlin...,60.0,10000.0,United States,4.983821,14.0,$18,$30,"[ad copy, about us page, headline, social medi...",1,24.000000
